<a href="https://colab.research.google.com/github/deepanrajm/deep_learning/blob/master/XAI/Video_AI_Tracking_Activity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Video AI: Visualization, Object Tracking & Activity Recognition


## 0. Setup

In [ ]:
!pip -q install ultralytics lap
import os, cv2, math, time, urllib.request, collections
import numpy as np
import matplotlib.pyplot as plt
import torch
from IPython.display import HTML, display
from base64 import b64encode

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device, '| Torch:', torch.__version__, '| OpenCV:', cv2.__version__)

In [ ]:
# --- Download sample videos (Intel IoT DevKit sample videos, public on GitHub) ---
BASE = 'https://github.com/intel-iot-devkit/sample-videos/raw/master/'
VIDEOS = {
    'people': BASE + 'people-detection.mp4',               # people walking in a room
    'street': BASE + 'person-bicycle-car-detection.mp4',   # street with people, bikes, cars
    'cars':   BASE + 'car-detection.mp4',                  # traffic
}
os.makedirs('videos', exist_ok=True); os.makedirs('outputs', exist_ok=True)
for name, url in VIDEOS.items():
    path = f'videos/{name}.mp4'
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
    print(name, '->', path, f'{os.path.getsize(path)/1e6:.1f} MB')

In [ ]:
# --- Helper functions ---
def video_info(path):
    cap = cv2.VideoCapture(path)
    info = dict(fps=cap.get(cv2.CAP_PROP_FPS), frames=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
                width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
    info['duration_s'] = round(info['frames'] / max(info['fps'], 1e-6), 1)
    cap.release()
    return info

def read_frames(path, max_frames=None, step=1):
    """Return a list of BGR frames."""
    cap, frames, i = cv2.VideoCapture(path), [], 0
    while True:
        ok, f = cap.read()
        if not ok or (max_frames and len(frames) >= max_frames): break
        if i % step == 0: frames.append(f)
        i += 1
    cap.release()
    return frames

def show_video(path, width=720):
    """Re-encode to H.264 (browser friendly) and embed in the notebook."""
    out = path.rsplit('.', 1)[0] + '_h264.mp4'
    os.system(f'ffmpeg -y -loglevel error -i "{path}" -vcodec libx264 -pix_fmt yuv420p "{out}"')
    data = b64encode(open(out, 'rb').read()).decode()
    display(HTML(f'<video width={width} controls loop><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'))

def process_video(src, dst, fn, max_frames=None, start=0):
    """Apply fn(frame, idx) -> output_frame to every frame (from `start`) and write a new video."""
    cap = cv2.VideoCapture(src); fps = cap.get(cv2.CAP_PROP_FPS); writer = None; i = 0
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    while True:
        ok, frame = cap.read()
        if not ok or (max_frames and i >= max_frames): break
        out = fn(frame, i)
        if writer is None:
            h, w = out.shape[:2]
            writer = cv2.VideoWriter(dst, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
        writer.write(out); i += 1
    cap.release(); writer.release()
    return dst

def imshow(img_bgr, title='', size=(10, 6)):
    plt.figure(figsize=size); plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)); plt.title(title); plt.axis('off'); plt.show()

def color_for(track_id):
    rng = np.random.RandomState(int(track_id) * 7 + 3)
    return tuple(int(c) for c in rng.randint(60, 255, 3))

def draw_box(img, box, label, color):
    x1, y1, x2, y2 = map(int, box)
    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
    cv2.rectangle(img, (x1, y1 - th - 6), (x1 + tw + 4, y1), color, -1)
    cv2.putText(img, label, (x1 + 2, y1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)

def put_banner(img, text):
    cv2.rectangle(img, (0, 0), (img.shape[1], 28), (30, 30, 30), -1)
    cv2.putText(img, text, (8, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)

for k in VIDEOS: print(k, video_info(f'videos/{k}.mp4'))

## Part 1 — Video basics and visualization

A video = a sequence of frames + time. **FPS** = frames per second, so `frames / fps = duration`.
The extra signal in video is **temporal continuity**: objects move only a little between consecutive frames.

In [ ]:
show_video('videos/people.mp4')

### 1.1 Frame grid (contact sheet) — the whole story in one image

In [ ]:
def frame_grid(path, n=12, cols=4):
    info = video_info(path); idxs = np.linspace(0, info['frames'] - 1, n).astype(int)
    cap = cv2.VideoCapture(path); rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 2.5))
    for ax, i in zip(axes.flat, idxs):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i)); ok, f = cap.read()
        ax.imshow(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)); ax.set_title(f'frame {i}  ({i/info["fps"]:.1f}s)', fontsize=9); ax.axis('off')
    cap.release(); plt.tight_layout(); plt.show()

frame_grid('videos/street.mp4')

### 1.2 Frame differencing — the simplest motion detector
`|frame(t) − frame(t−1)|` → threshold → motion mask. Fails with camera motion and lighting changes.

In [ ]:
frames = read_frames('videos/people.mp4', max_frames=400)
t = 375
g1 = cv2.cvtColor(frames[t - 1], cv2.COLOR_BGR2GRAY); g2 = cv2.cvtColor(frames[t], cv2.COLOR_BGR2GRAY)
diff = cv2.absdiff(g2, g1)
_, mask = cv2.threshold(cv2.GaussianBlur(diff, (5, 5), 0), 20, 255, cv2.THRESH_BINARY)
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].imshow(cv2.cvtColor(frames[t], cv2.COLOR_BGR2RGB)); ax[0].set_title('Frame t')
ax[1].imshow(diff, cmap='inferno'); ax[1].set_title('|frame t − frame t−1|')
ax[2].imshow(mask, cmap='gray'); ax[2].set_title('Thresholded motion mask')
for a in ax: a.axis('off')
plt.show()

### 1.3 Background subtraction (MOG2)
Each pixel is modelled as a mixture of Gaussians over time; pixels that don't fit the model are foreground. Classic CCTV technique.

In [ ]:
mog = cv2.createBackgroundSubtractorMOG2(history=200, varThreshold=25, detectShadows=True)
def bg_sub(frame, i):
    fg = mog.apply(frame)
    fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    fg_color = cv2.cvtColor(fg, cv2.COLOR_GRAY2BGR)
    contours, _ = cv2.findContours((fg == 255).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    out = frame.copy()
    for c in contours:
        if cv2.contourArea(c) > 400:
            x, y, w, h = cv2.boundingRect(c); cv2.rectangle(out, (x, y), (x + w, y + h), (0, 255, 0), 2)
    both = np.hstack([out, fg_color]); put_banner(both, 'Background subtraction (MOG2): blobs  |  foreground mask (grey = shadow)')
    return both
show_video(process_video('videos/people.mp4', 'outputs/bg_sub.mp4', bg_sub), width=900)

### 1.4 Dense optical flow (Farnebäck)
A motion vector for **every pixel**. Visualization: direction → colour (hue), speed → brightness.

In [ ]:
prev_gray = None
def flow_viz(frame, i):
    global prev_gray
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    if prev_gray is None: prev_gray = gray
    flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    prev_gray = gray
    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    hsv = np.zeros_like(frame); hsv[..., 0] = ang * 180 / np.pi / 2; hsv[..., 1] = 255
    hsv[..., 2] = np.clip(mag * 25, 0, 255)          # fixed scale, so static frames stay dark
    flow_bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    # also draw sparse arrows
    arrows = frame.copy()
    for y in range(10, frame.shape[0], 20):
        for x in range(10, frame.shape[1], 20):
            dx, dy = flow[y, x]
            if dx * dx + dy * dy > 1:
                cv2.arrowedLine(arrows, (x, y), (int(x + dx * 3), int(y + dy * 3)), (0, 255, 255), 1, tipLength=0.4)
    both = np.hstack([arrows, flow_bgr]); put_banner(both, 'Optical flow: motion vectors  |  HSV (hue = direction, brightness = speed)')
    return both
show_video(process_video('videos/street.mp4', 'outputs/flow.mp4', flow_viz, start=140, max_frames=160), width=900)

### 1.5 Motion heatmap — where does activity happen?

In [ ]:
mog_h = cv2.createBackgroundSubtractorMOG2(history=200, varThreshold=25, detectShadows=False)
frames = read_frames('videos/street.mp4')
acc = np.zeros(frames[0].shape[:2], np.float32)
for f in frames:
    acc += (mog_h.apply(f) > 0).astype(np.float32)
heat = cv2.applyColorMap(cv2.normalize(acc, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8), cv2.COLORMAP_JET)
imshow(cv2.addWeighted(frames[len(frames) // 2], 0.55, heat, 0.45, 0), 'Motion heatmap (accumulated foreground over the whole video)')

## Part 2 — Object detection on video (YOLO11)
Run the detector on every frame. Notice: boxes and classes, **but no identity** — the model has no idea that the person in frame 10 is the same person in frame 11.

In [ ]:
from ultralytics import YOLO
det_model = YOLO('yolo11n.pt')    # auto-downloads (~5.6 MB). Try yolo11s.pt / yolo11m.pt for better accuracy.
frame = read_frames('videos/street.mp4', max_frames=561)[-1]   # a frame with a car, a cyclist and a person
res = det_model.predict(frame, conf=0.3, verbose=False)[0]
imshow(res.plot(), f'YOLO11n: {len(res.boxes)} objects detected in one frame')
print({det_model.names[int(c)]: int((res.boxes.cls == c).sum()) for c in res.boxes.cls.unique()})

In [ ]:
def detect_only(frame, i):
    r = det_model.predict(frame, conf=0.3, verbose=False)[0]
    out = r.plot(); put_banner(out, f'Detection only (frame {i}) - no IDs, no memory between frames')
    return out
show_video(process_video('videos/street.mp4', 'outputs/detect_only.mp4', detect_only, start=480, max_frames=160))

## Part 3 — The tracking ladder

**Tracking-by-detection loop (every frame):** Detect → Predict (motion model) → Associate (cost matrix + matching) → Update → Manage track life-cycle.

To compare trackers fairly we first **cache the YOLO detections** for the `people` video, then feed the *same* boxes to every tracker we write ourselves.

In [12]:
def cache_detections(src, classes=(0,), conf=0.3):
    dets = []
    for r in det_model.predict(src, stream=True, classes=list(classes), conf=conf, verbose=False):
        b = r.boxes
        dets.append(np.hstack([b.xyxy.cpu().numpy(), b.conf.cpu().numpy()[:, None]]) if len(b) else np.zeros((0, 5)))
    return dets

people_dets = cache_detections('videos/people.mp4')
print('frames:', len(people_dets), '| avg persons/frame:', np.mean([len(d) for d in people_dets]).round(2))

frames: 596 | avg persons/frame: 0.69


In [13]:
# ---------- Shared utilities for trackers ----------
from scipy.optimize import linear_sum_assignment

def iou_matrix(a, b):
    """Pairwise IoU between boxes a (N,4) and b (M,4) in xyxy format."""
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)

def run_custom_tracker(tracker, dets, src, dst, title):
    """Feed cached detections to a tracker, draw IDs + trails, return the set of IDs ever shown."""
    cap = cv2.VideoCapture(src); fps = cap.get(cv2.CAP_PROP_FPS); writer = None
    trails = collections.defaultdict(lambda: collections.deque(maxlen=30)); all_ids = set()
    for i, d in enumerate(dets):
        ok, frame = cap.read()
        if not ok: break
        tracks = tracker.update(d[:, :4] if len(d) else np.zeros((0, 4)))
        for tid, box in tracks:
            all_ids.add(tid); c = color_for(tid)
            cx, cy = int((box[0] + box[2]) / 2), int(box[3])
            trails[tid].append((cx, cy))
            pts = list(trails[tid])
            for p, q in zip(pts[:-1], pts[1:]): cv2.line(frame, p, q, c, 2)
            draw_box(frame, box, f'ID {tid}', c)
        put_banner(frame, f'{title} | frame {i} | unique IDs so far: {len(all_ids)}')
        if writer is None:
            writer = cv2.VideoWriter(dst, cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame.shape[1], frame.shape[0]))
        writer.write(frame)
    cap.release(); writer.release()
    return all_ids

### 3.1 Centroid tracker
Match each old object to the **nearest new centre** (greedy, within `max_dist` pixels). No motion model, no box size.
Watch for ID swaps when people cross.

In [ ]:
class CentroidTracker:
    def __init__(self, max_dist=60, max_missed=10):
        self.max_dist, self.max_missed = max_dist, max_missed
        self.next_id, self.tracks = 1, {}          # id -> {'box', 'missed'}

    @staticmethod
    def centre(b): return (b[..., :2] + b[..., 2:4]) / 2

    def update(self, boxes):
        ids = list(self.tracks.keys()); matched_t, matched_d = set(), set()
        if ids and len(boxes):
            tc = np.array([self.centre(self.tracks[t]['box']) for t in ids])
            D = np.linalg.norm(tc[:, None] - self.centre(boxes)[None], axis=2)
            for r, c in zip(*np.unravel_index(np.argsort(D, axis=None), D.shape)):   # greedy: closest pairs first
                if r in matched_t or c in matched_d or D[r, c] > self.max_dist: continue
                self.tracks[ids[r]].update(box=boxes[c], missed=0); matched_t.add(r); matched_d.add(c)
        for r, t in enumerate(ids):
            if r not in matched_t:
                self.tracks[t]['missed'] += 1
                if self.tracks[t]['missed'] > self.max_missed: del self.tracks[t]
        for c in range(len(boxes)):
            if c not in matched_d:
                self.tracks[self.next_id] = {'box': boxes[c], 'missed': 0}; self.next_id += 1
        return [(t, v['box']) for t, v in self.tracks.items() if v['missed'] == 0]

ids_centroid = run_custom_tracker(CentroidTracker(), people_dets, 'videos/people.mp4', 'outputs/track_centroid.mp4', 'Centroid tracker')
print('Unique IDs (Centroid):', len(ids_centroid))
show_video('outputs/track_centroid.mp4')

### 3.2 IoU tracker with Hungarian matching
Cost = `1 − IoU(old box, new box)`. The **Hungarian algorithm** (`linear_sum_assignment`) finds the globally best one-to-one matching.
Still no prediction, so a single missed detection or fast motion breaks the track.

In [ ]:
class IoUTracker:
    def __init__(self, iou_thresh=0.3, max_missed=5):
        self.iou_thresh, self.max_missed = iou_thresh, max_missed
        self.next_id, self.tracks = 1, {}

    def update(self, boxes):
        ids = list(self.tracks.keys()); matched_t, matched_d = set(), set()
        if ids and len(boxes):
            iou = iou_matrix(np.array([self.tracks[t]['box'] for t in ids]), boxes)
            rows, cols = linear_sum_assignment(1 - iou)                 # Hungarian algorithm
            for r, c in zip(rows, cols):
                if iou[r, c] >= self.iou_thresh:
                    self.tracks[ids[r]].update(box=boxes[c], missed=0); matched_t.add(r); matched_d.add(c)
        for r, t in enumerate(ids):
            if r not in matched_t:
                self.tracks[t]['missed'] += 1
                if self.tracks[t]['missed'] > self.max_missed: del self.tracks[t]
        for c in range(len(boxes)):
            if c not in matched_d:
                self.tracks[self.next_id] = {'box': boxes[c], 'missed': 0}; self.next_id += 1
        return [(t, v['box']) for t, v in self.tracks.items() if v['missed'] == 0]

ids_iou = run_custom_tracker(IoUTracker(), people_dets, 'videos/people.mp4', 'outputs/track_iou.mp4', 'IoU + Hungarian tracker')
print('Unique IDs (IoU):', len(ids_iou))
show_video('outputs/track_iou.mp4')

###  SORT from scratch (Kalman filter + Hungarian)
Each track owns a **Kalman filter** with state `[cx, cy, s, r, vx, vy, vs]` (centre, area, aspect ratio + velocities).

* **Predict:** constant-velocity motion → where should the box be now? Uncertainty `P` grows.
* **Associate:** IoU between *predicted* boxes and detections → Hungarian.
* **Update:** blend prediction and detection using the Kalman gain `K`.
* Lost tracks keep *coasting* on their prediction for `max_age` frames → they survive short occlusions.

In [ ]:
class KalmanBoxTracker:
    count = 0
    def __init__(self, box):
        # State x = [cx, cy, s, r, vx, vy, vs]^T ; measurement z = [cx, cy, s, r]^T
        self.F = np.eye(7); self.F[0, 4] = self.F[1, 5] = self.F[2, 6] = 1     # constant velocity model
        self.H = np.eye(4, 7)                                                  # we measure position only
        self.R = np.diag([1., 1., 10., 10.])                                   # measurement noise
        self.P = np.diag([10., 10., 10., 10., 1e4, 1e4, 1e4])                  # high initial velocity uncertainty
        self.Q = np.diag([1., 1., 1., 1., 1e-2, 1e-2, 1e-4])                  # process noise
        self.x = np.zeros((7, 1)); self.x[:4] = self.to_z(box)
        KalmanBoxTracker.count += 1; self.id = KalmanBoxTracker.count
        self.time_since_update, self.hits, self.hit_streak = 0, 1, 1

    @staticmethod
    def to_z(b):
        w, h = b[2] - b[0], b[3] - b[1]
        return np.array([[b[0] + w / 2], [b[1] + h / 2], [w * h], [w / max(h, 1e-6)]])

    def box(self):
        cx, cy, s, r = self.x[:4, 0]; w = math.sqrt(max(s * r, 1e-6)); h = s / w
        return np.array([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2])

    def predict(self):
        if self.x[2, 0] + self.x[6, 0] <= 0: self.x[6, 0] = 0                  # area can't go negative
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q
        if self.time_since_update > 0: self.hit_streak = 0
        self.time_since_update += 1
        return self.box()

    def update(self, box):
        z = self.to_z(box)
        y = z - self.H @ self.x                                  # innovation (measurement - prediction)
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)                 # Kalman gain
        self.x = self.x + K @ y
        self.P = (np.eye(7) - K @ self.H) @ self.P
        self.time_since_update = 0; self.hits += 1; self.hit_streak += 1


class SORT:
    def __init__(self, max_age=12, min_hits=3, iou_thresh=0.3):
        self.max_age, self.min_hits, self.iou_thresh = max_age, min_hits, iou_thresh
        self.trackers, self.frame = [], 0
        KalmanBoxTracker.count = 0

    def update(self, boxes):
        self.frame += 1
        preds = np.array([t.predict() for t in self.trackers]) if self.trackers else np.zeros((0, 4))
        matched_t, matched_d = set(), set()
        if len(preds) and len(boxes):
            iou = iou_matrix(preds, boxes)
            for r, c in zip(*linear_sum_assignment(-iou)):
                if iou[r, c] >= self.iou_thresh:
                    self.trackers[r].update(boxes[c]); matched_t.add(r); matched_d.add(c)
        for c in range(len(boxes)):
            if c not in matched_d: self.trackers.append(KalmanBoxTracker(boxes[c]))
        self.trackers = [t for t in self.trackers if t.time_since_update <= self.max_age]
        return [(t.id, t.box()) for t in self.trackers
                if t.time_since_update == 0 and (t.hit_streak >= self.min_hits or self.frame <= self.min_hits)]

ids_sort = run_custom_tracker(SORT(), people_dets, 'videos/people.mp4', 'outputs/track_sort.mp4', 'SORT (Kalman + Hungarian)')
print('Unique IDs (SORT):', len(ids_sort))
show_video('outputs/track_sort.mp4')

#### See the Kalman filter at work
We hide the detections of one track for 15 frames (a simulated occlusion). The filter keeps predicting the box, and the track is re-matched when detections return.

In [ ]:
kf = KalmanBoxTracker(np.array([100, 100, 140, 200.]))
true_x, meas, est, hidden = [], [], [], range(25, 40)
for t in range(60):
    gt = 100 + 4 * t + 10 * math.sin(t / 6)                      # true moving x position
    true_x.append(gt); kf.predict()
    if t not in hidden:
        z = gt + np.random.randn() * 3
        kf.update(np.array([z, 100, z + 40, 200.])); meas.append((t, z + 20))
    est.append(kf.box()[0] + 20)
plt.figure(figsize=(10, 4))
plt.plot(np.array(true_x) + 20, 'k--', label='true centre x')
plt.scatter(*zip(*meas), s=12, c='tab:orange', label='noisy detections')
plt.plot(est, 'tab:blue', lw=2, label='Kalman estimate')
plt.axvspan(25, 39, color='grey', alpha=0.2, label='occlusion (no detections)')
plt.xlabel('frame'); plt.ylabel('x (pixels)'); plt.legend(); plt.title('Kalman filter: smoothing + prediction through occlusion'); plt.show()

### 3.4 — Ultralytics trackers: ByteTrack and BoT-SORT
* **ByteTrack** — match high-confidence boxes first, then use **low-confidence boxes** to rescue partially occluded tracks.
* **BoT-SORT** (Ultralytics default) — ByteTrack + better Kalman state + **camera motion compensation** + optional **Re-ID** appearance features.

Let's look at the configs, then run both.

In [ ]:
import ultralytics, yaml
cfg_dir = os.path.join(os.path.dirname(ultralytics.__file__), 'cfg', 'trackers')
print(open(os.path.join(cfg_dir, 'bytetrack.yaml')).read())
print('-' * 60)
print(open(os.path.join(cfg_dir, 'botsort.yaml')).read())

In [ ]:
def run_ultralytics_tracker(src, dst, tracker_cfg, classes=(0,), title='', model_name='yolo11n.pt'):
    model = YOLO(model_name)       # fresh model => fresh tracker state
    fps = video_info(src)['fps']; writer = None; all_ids = set()
    trails = collections.defaultdict(lambda: collections.deque(maxlen=30))
    for i, r in enumerate(model.track(src, tracker=tracker_cfg, classes=list(classes), stream=True, persist=True, verbose=False)):
        frame = r.orig_img.copy()
        if r.boxes.id is not None:
            for tid, box, cls in zip(r.boxes.id.int().cpu().tolist(), r.boxes.xyxy.cpu().numpy(), r.boxes.cls.int().cpu().tolist()):
                all_ids.add(tid); c = color_for(tid)
                trails[tid].append((int((box[0] + box[2]) / 2), int(box[3])))
                pts = list(trails[tid])
                for p, q in zip(pts[:-1], pts[1:]): cv2.line(frame, p, q, c, 2)
                draw_box(frame, box, f'{model.names[cls]} {tid}', c)
        put_banner(frame, f'{title} | frame {i} | unique IDs so far: {len(all_ids)}')
        if writer is None:
            writer = cv2.VideoWriter(dst, cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame.shape[1], frame.shape[0]))
        writer.write(frame)
    writer.release()
    return all_ids

ids_byte = run_ultralytics_tracker('videos/people.mp4', 'outputs/track_bytetrack.mp4', 'bytetrack.yaml', title='ByteTrack')
print('Unique IDs (ByteTrack):', len(ids_byte))
show_video('outputs/track_bytetrack.mp4')

In [ ]:
ids_bot = run_ultralytics_tracker('videos/people.mp4', 'outputs/track_botsort.mp4', 'botsort.yaml', title='BoT-SORT')
print('Unique IDs (BoT-SORT):', len(ids_bot))
show_video('outputs/track_botsort.mp4')

### 3.5 Tuning a tracker: custom BoT-SORT with longer memory + Re-ID
`track_buffer` = how many frames a lost track is remembered. `with_reid` adds appearance matching (needs a recent Ultralytics version; we fall back gracefully if unsupported).

In [ ]:
cfg = yaml.safe_load(open(os.path.join(cfg_dir, 'botsort.yaml')))
cfg.update(track_buffer=60, with_reid=True)
cfg.setdefault('model', 'auto')
yaml.safe_dump(cfg, open('my_botsort.yaml', 'w'))
try:
    ids_bot_reid = run_ultralytics_tracker('videos/people.mp4', 'outputs/track_botsort_reid.mp4', 'my_botsort.yaml', title='BoT-SORT + ReID, buffer 60')
except Exception as e:
    print('Re-ID not supported in this version -> using track_buffer=60 only:', e)
    cfg['with_reid'] = False; yaml.safe_dump(cfg, open('my_botsort.yaml', 'w'))
    ids_bot_reid = run_ultralytics_tracker('videos/people.mp4', 'outputs/track_botsort_reid.mp4', 'my_botsort.yaml', title='BoT-SORT, buffer 60')
print('Unique IDs (custom BoT-SORT):', len(ids_bot_reid))
show_video('outputs/track_botsort_reid.mp4')

### 3.6 Compare the ladder
Same video, so the true number of people is fixed. **Fewer unique IDs ≈ fewer broken / switched tracks** (a rough proxy for IDF1 — real benchmarks use ground truth and HOTA/IDF1/MOTA).

In [ ]:
results = {'Centroid': len(ids_centroid), 'IoU + Hungarian': len(ids_iou), 'SORT (scratch)': len(ids_sort),
           'ByteTrack': len(ids_byte), 'BoT-SORT': len(ids_bot), 'BoT-SORT tuned': len(ids_bot_reid)}
plt.figure(figsize=(10, 4)); bars = plt.bar(results.keys(), results.values(), color=['#999', '#999', '#5a9', '#2a7', '#174', '#063'])
plt.bar_label(bars); plt.ylabel('unique IDs created'); plt.title('Identity fragmentation across trackers (lower is better)')
plt.xticks(rotation=15); plt.show()

## Part 4 — Activity recognition
A single frame cannot tell *walking* from *standing*; we need **time**.

| Approach | Idea | Here |
|---|---|---|
| Pose / skeleton | 17 body keypoints per person → rules or a graph network (ST-GCN) | YOLO11-pose + tracking + rules |
| 3D CNN | Convolution over time × height × width | R3D-18 (Kinetics-400, 400 actions) |
| Video Transformer | Space-time attention over patches | VideoMAE (Kinetics-400) |

### 4.1 Pose-based activity (YOLO11-pose + tracking + simple rules)
COCO keypoints: 0 nose · 5/6 shoulders · 7/8 elbows · 9/10 wrists · 11/12 hips · 13/14 knees · 15/16 ankles.

In [ ]:
pose_model = YOLO('yolo11n-pose.pt')
SKELETON = [(5, 7), (7, 9), (6, 8), (8, 10), (5, 6), (5, 11), (6, 12), (11, 12), (11, 13), (13, 15), (12, 14), (14, 16), (0, 5), (0, 6)]

def joint_angle(a, b, c):
    ba, bc = a - b, c - b
    cos = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))

def classify_pose(kp, box, speed_px_s):
    xy, conf = kp[:, :2], kp[:, 2]
    ok = lambda *idx: all(conf[j] > 0.5 for j in idx)
    w, h = box[2] - box[0], box[3] - box[1]
    if w > 1.3 * h: return 'LYING / FALLEN'
    if ok(0, 9, 10) and xy[9, 1] < xy[0, 1] and xy[10, 1] < xy[0, 1]: return 'HANDS UP'
    if (ok(0, 9) and xy[9, 1] < xy[0, 1]) or (ok(0, 10) and xy[10, 1] < xy[0, 1]): return 'ONE HAND UP'
    knees = [joint_angle(xy[a], xy[b], xy[c]) for a, b, c in [(11, 13, 15), (12, 14, 16)] if ok(a, b, c)]
    if knees and np.mean(knees) < 120: return 'SITTING / BENDING'
    return 'WALKING' if speed_px_s > 20 else 'STANDING'

src = 'videos/people.mp4'; fps = video_info(src)['fps']
centres = collections.defaultdict(lambda: collections.deque(maxlen=8)); activity_log = collections.Counter()
def pose_activity(frame, i):
    r = pose_model.track(frame, persist=True, verbose=False, tracker='bytetrack.yaml')[0]
    out = frame.copy()
    if r.boxes.id is not None and r.keypoints is not None:
        for tid, box, kp in zip(r.boxes.id.int().cpu().tolist(), r.boxes.xyxy.cpu().numpy(), r.keypoints.data.cpu().numpy()):
            centres[tid].append(((box[0] + box[2]) / 2, box[3]))
            speed = 0
            if len(centres[tid]) > 2:
                (x0, y0), (x1, y1) = centres[tid][0], centres[tid][-1]
                speed = math.hypot(x1 - x0, y1 - y0) / ((len(centres[tid]) - 1) / fps)
            act = classify_pose(kp, box, speed); activity_log[act] += 1
            for a, b in SKELETON:
                if kp[a, 2] > 0.5 and kp[b, 2] > 0.5:
                    cv2.line(out, tuple(kp[a, :2].astype(int)), tuple(kp[b, :2].astype(int)), (0, 255, 0), 2)
            draw_box(out, box, f'ID {tid}: {act}', color_for(tid))
    put_banner(out, f'Pose-based activity recognition | frame {i}')
    return out
show_video(process_video(src, 'outputs/pose_activity.mp4', pose_activity))
print('Activity frequency (person-frames):', dict(activity_log))

### 5.2 3D-CNN: R3D-18 pretrained on Kinetics-400
Input: a **clip** of 16 frames at 112×112 → tensor `(1, 3, 16, 112, 112)`. Output: 400 action classes.
Sample clip from the Hugging Face hub (`eating_spaghetti.mp4`, used in the VideoMAE docs).

In [ ]:
from torchvision.models.video import r3d_18, R3D_18_Weights
from huggingface_hub import hf_hub_download

try:
    action_clip = hf_hub_download(repo_id='nielsr/video-demo', filename='eating_spaghetti.mp4', repo_type='dataset')
except Exception as e:
    print('HF download failed, using local people video instead:', e); action_clip = 'videos/people.mp4'
print('Clip:', action_clip, video_info(action_clip))

weights = R3D_18_Weights.KINETICS400_V1
r3d = r3d_18(weights=weights).eval().to(device)
r3d_tf = weights.transforms()
KINETICS = weights.meta['categories']

def sample_clip(frames_bgr, n=16):
    idx = np.linspace(0, len(frames_bgr) - 1, n).astype(int)
    return [cv2.cvtColor(frames_bgr[i], cv2.COLOR_BGR2RGB) for i in idx]

@torch.no_grad()
def r3d_predict(rgb_frames, topk=5):
    x = torch.from_numpy(np.stack(rgb_frames)).permute(0, 3, 1, 2)       # (T, C, H, W)
    x = r3d_tf(x).unsqueeze(0).to(device)                                # (1, C, T, H, W)
    probs = r3d(x).softmax(-1)[0]
    p, i = probs.topk(topk)
    return [(KINETICS[j], float(q)) for q, j in zip(p.cpu(), i.cpu())]

clip_frames = read_frames(action_clip)
clip16 = sample_clip(clip_frames)
fig, ax = plt.subplots(1, 8, figsize=(18, 2.5))
for a, f in zip(ax, clip16[::2]): a.imshow(f); a.axis('off')
plt.suptitle('The 16-frame clip the 3D-CNN sees (every 2nd frame shown)'); plt.show()
for name, p in r3d_predict(clip16): print(f'{p:6.1%}  {name}')

### 5.3 Video Transformer: VideoMAE (Kinetics-400)
Patches are taken over space **and** time; attention relates any patch to any other. Pre-trained by masking ~90% of patches and reconstructing them.

In [ ]:
from transformers import VideoMAEImageProcessor, VideoMAEForVideoClassification
vm_proc = VideoMAEImageProcessor.from_pretrained('MCG-NJU/videomae-base-finetuned-kinetics')
vm = VideoMAEForVideoClassification.from_pretrained('MCG-NJU/videomae-base-finetuned-kinetics').to(device).eval()
with torch.no_grad():
    inputs = vm_proc(list(clip16), return_tensors='pt').to(device)
    probs = vm(**inputs).logits.softmax(-1)[0]
p, i = probs.topk(5)
for q, j in zip(p.cpu(), i.cpu()): print(f'{float(q):6.1%}  {vm.config.id2label[int(j)]}')

## Part 6 — GenAI for video (pointers + mini demo)
* **Keyframes + captioning** → turn video into text (BLIP below).
* **Open-vocabulary detection** (YOLO-World): detect objects described in words, no retraining.
* **Event log + LLM**: tracking output becomes structured text an LLM can summarise or answer questions about.
* Also discuss: video-LLMs (Qwen2.5-VL, Gemini, Video-LLaVA), text-to-video (Sora, Veo, Wan, CogVideoX), synthetic training data.

In [21]:
# 6.1 Keyframe extraction by colour-histogram change
def keyframes(path, thresh=0.25, max_k=6):
    frames, keys, prev = read_frames(path, step=3), [], None
    for i, f in enumerate(frames):
        h = cv2.calcHist([cv2.cvtColor(f, cv2.COLOR_BGR2HSV)], [0, 1], None, [32, 32], [0, 180, 0, 256]); cv2.normalize(h, h)
        if prev is None or cv2.compareHist(prev, h, cv2.HISTCMP_BHATTACHARYYA) > thresh:
            keys.append(f); prev = h
    idx = np.linspace(0, len(keys) - 1, min(max_k, len(keys))).astype(int)
    return [keys[i] for i in idx]

kfs = keyframes('videos/street.mp4')
print('Keyframes selected:', len(kfs))

Keyframes selected: 5


In [ ]:
# 6.2 Caption each keyframe with BLIP (image-to-text model)
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
blip_proc = BlipProcessor.from_pretrained('Salesforce/blip-image-captioning-base')
blip = BlipForConditionalGeneration.from_pretrained('Salesforce/blip-image-captioning-base').to(device)
captions = []
fig, ax = plt.subplots(1, len(kfs), figsize=(4 * len(kfs), 3.5))
for a, f in zip(np.atleast_1d(ax), kfs):
    img = Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    with torch.no_grad():
        out = blip.generate(**blip_proc(images=img, return_tensors='pt').to(device), max_new_tokens=30)
    cap = blip_proc.decode(out[0], skip_special_tokens=True); captions.append(cap)
    a.imshow(img); a.set_title(cap, fontsize=9, wrap=True); a.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# 6.3 Open-vocabulary detection with YOLO-World (describe what to find in words)
try:
    from ultralytics import YOLOWorld
    world = YOLOWorld('yolov8s-worldv2.pt')
    world.set_classes(['person', 'bicycle', 'car', 'traffic light', 'person carrying a bag'])
    f = read_frames('videos/street.mp4', max_frames=561)[-1]
    imshow(world.predict(f, conf=0.15, verbose=False)[0].plot(), 'YOLO-World: classes given as text prompts')
except Exception as e:
    print('YOLO-World demo skipped:', e)

In [ ]:
# 6.4 Build an LLM-ready prompt from video analytics (paste into any LLM, or call an API)
prompt = f"""You are a video analytics assistant.
Scene captions (keyframes, in order): {captions}
Traffic events from the tracker: {events[:20]}
Line-crossing totals: {counts}
Pose-activity frequency in the indoor video: {dict(activity_log)}

Write a 5-line incident-style summary of what happened, then list anything unusual."""
print(prompt)